# stepzero — Quickstart

A 5-minute tour of all six tasks.

In [ ]:
import stepzero as sz
print(sz.__version__)

## 1. Classification — Iris

In [ ]:
from sklearn.datasets import load_iris
X, y = load_iris(return_X_y=True, as_frame=True)

result = sz.classification(X, y)
print(result)
print()
print(result.headroom)


In [ ]:
# All models compared
for s in result.scores:
    print(f"  {s.name:<15} {s.metric} = {s.score:.3f}")


In [ ]:
# Feature importance from the winning model
result.feature_importance.plot(kind="barh", title="Feature importance")


## 2. Regression — Diabetes

In [ ]:
from sklearn.datasets import load_diabetes
X, y = load_diabetes(return_X_y=True, as_frame=True)

result = sz.regression(X, y)
print(result)
print()
print(result.headroom)


In [ ]:
result.feature_importance.plot(kind="barh", title="Feature importance (normalized)")


## 3. Forecasting — Synthetic trend + seasonality

In [ ]:
import numpy as np
import pandas as pd

# 4 years of monthly data: upward trend + yearly seasonality
idx = pd.date_range("2020-01", periods=48, freq="ME")
trend = np.linspace(100, 160, 48)
seasonal = 10 * np.sin(2 * np.pi * np.arange(48) / 12)
noise = np.random.default_rng(0).normal(0, 3, 48)
ts = pd.Series(trend + seasonal + noise, index=idx, name="value")

ts.plot(title="Training series", figsize=(10, 3))


In [ ]:
result = sz.forecasting(ts, horizon=12)
print(result)
print()
print(result.headroom)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ts.plot(ax=ax, label="observed")
result.forecast.plot(ax=ax, label=f"forecast ({result.best_model_name})", linestyle="--", marker="o")
ax.legend()
ax.set_title("stepzero forecast — 12-month horizon")


## 4. Anomaly detection — Synthetic sensor data

In [ ]:
rng = np.random.default_rng(42)
sensor = pd.Series(rng.normal(0, 1, 200), name="sensor")
sensor.iloc[[20, 80, 140, 180]] = [6, -7, 8, -6]  # inject 4 spikes

result = sz.anomaly_detection(sensor)
print(result)
print()
print(result.headroom)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
sensor.plot(ax=ax, label="signal", alpha=0.7)
sensor[result.anomalies].plot(ax=ax, style="rv", label="anomaly", markersize=10)
ax.legend()
ax.set_title(f"Anomaly detection ({result.method}, threshold={result.threshold:.2f})")


## 5. Text classification — 20 Newsgroups (2 categories)

In [ ]:
from sklearn.datasets import fetch_20newsgroups

cats = ["sci.space", "talk.politics.guns"]
train = fetch_20newsgroups(subset="train", categories=cats)

result = sz.text_classification(train.data, train.target, cv=3)
print(result)
print()
print(result.headroom)


In [ ]:
# Top discriminative terms per class
for cls, terms in result.top_features_per_class.items():
    label = train.target_names[int(cls)]
    print(f"  {label}: {', '.join(terms[:8])}")


## 6. Clustering — make_blobs

In [ ]:
from sklearn.datasets import make_blobs

X_blobs, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=0)

result = sz.clustering(X_blobs, k_range=(2, 8))
print(result)
print()
print(result.headroom)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: cluster assignments
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=result.labels, cmap="tab10", s=20)
axes[0].scatter(result.centers[:, 0], result.centers[:, 1], marker="X", s=200, c="black", label="centers")
axes[0].set_title(f"K-means, k={result.best_k}")
axes[0].legend()

# Right: silhouette score vs k
ks = [int(s.name.split("=")[1]) for s in result.scores]
sils = [s.score for s in result.scores]
axes[1].plot(ks, sils, marker="o")
axes[1].axvline(result.best_k, color="red", linestyle="--", label=f"best k={result.best_k}")
axes[1].set_xlabel("k")
axes[1].set_ylabel("silhouette score")
axes[1].set_title("Silhouette vs k")
axes[1].legend()
